# PromptGuard ML 탐지 평가 (Colab)

PromptGuard(promptguard-claude-demoV0)를 **체크아웃에 설정된 그대로** 평가한다. ML 모듈(`ml-v9`, `ml-qwen`)은 플러그인이 원래 제공하는 스위치 `PG_DETECTOR_ENABLE`로만 켠다. 플러그인 코드는 바꾸지 않는다.

순서: ① 설정 → ② 레포 받기 → ③ 패키지 → ④ gitleaks → ⑤ 데이터 → ⑥ 작은 확인 → ⑦ 전체 평가 → ⑧ 결과 표·저장.

준비물: Google Drive에 올린 데이터 압축 파일(`metric-detection/colab/README.md`의 "데이터 올리기" 참고). 런타임은 GPU(T4 이상)를 권장한다. Qwen이 GPU를 쓴다.

## ① 설정

In [ ]:
# 데이터: Drive에 올린 압축 파일. 1만 개 표본(pg-data-10k.tar.gz) 또는 전체(pg-data-full.tar.gz)
DATA_ARCHIVE = "/content/drive/MyDrive/promptguard/pg-data-10k.tar.gz"

# 켤 ML 모듈: "ml-v9", "ml-qwen", "ml-v9,ml-qwen"
ENABLE = "ml-v9"

# ML 없이 한 번 더 돌려 비교할지 (같은 데이터, 같은 코드)
RUN_BASELINE = True

# 평가할 데이터셋 파일 이름(stem). 비우면 전부
DATASETS = []  # 예: ["sessions_from_CredData", "sessions_from_privesc-llm-data"]

# 탐지기 실패 처리: "block"(Runtime V1: Final Guard가 요청을 막음, 기본) 또는 "pass"(옛 launcher: 원문 그대로)
ON_FAILURE = "block"

# ml-v9 실행 장치. None이면 detector.json 그대로("cpu"). GPU 런타임이면 "cuda"로 바꿔도 된다(이 Colab 사본에서만)
V9_DEVICE = None

# 결과를 복사할 Drive 폴더 (결과 파일에는 비밀값이 들어 있다. 공유 폴더에 두지 않는다)
OUT_DRIVE = "/content/drive/MyDrive/promptguard/results"

# 받을 브랜치
BRANCHES = {
    "promptguard-data": "main",
    "promptguard-claude-demoV0": "main",
    "promptguard-model": "main",
}

## ② 레포 받기

세 레포를 `/content` 아래에 **나란히** 둔다. 채점기는 옆의 `promptguard-claude-demoV0`를 찾고, 플러그인은 `src/detector` 기준 `../../../promptguard-model`에서 모델을 찾는다. 모델 가중치는 Git LFS로 받는다.

In [ ]:
import os, subprocess, shlex

def sh(cmd, env=None, check=True):
    print("$", cmd, flush=True)
    return subprocess.run(cmd, shell=True, check=check, env=env)

if subprocess.run("git lfs version", shell=True, capture_output=True).returncode != 0:
    sh("apt-get -qq install -y git-lfs")
sh("git lfs install")
for name, branch in BRANCHES.items():
    path = f"/content/{name}"
    if not os.path.isdir(path):
        sh(f"git clone --depth 1 -b {branch} https://github.com/capstone-privai/{name}.git {path}")
    else:
        sh(f"git -C {path} fetch --depth 1 origin {branch} && git -C {path} checkout -q FETCH_HEAD")
sh("cd /content/promptguard-model && git lfs pull")

weights = {
    "ml-v9": "/content/promptguard-model/models/span-detector-v9/model.safetensors",
    "ml-qwen": "/content/promptguard-model/models/qwen-span-v2/adapter_model.safetensors",
}
for name in [n.strip() for n in ENABLE.split(",") if n.strip()]:
    size = os.path.getsize(weights[name])
    print(f"{name}: {size / 1e6:.1f} MB")
    assert size > 1_000_000, f"{name} 가중치가 LFS 포인터 상태다. git lfs pull을 확인한다"
for name in BRANCHES:
    print(name, subprocess.run(f"git -C /content/{name} log --oneline -1", shell=True, capture_output=True, text=True).stdout.strip())

## ③ 패키지

`credsweeper`는 플러그인이 고정한 버전(1.19.0)이어야 한다. 다르면 채점기가 실행을 거부한다. `torch`는 Colab에 이미 있다. 설치 뒤 numpy 등이 바뀌었다는 경고가 나오면 **런타임 → 세션 다시 시작** 후 ①부터 다시 실행한다(②는 이미 받았으면 건너뛴다).

In [ ]:
sh('pip -q install "credsweeper==1.19.0" "transformers>=4.51,<5" "peft>=0.13,<1" "PyYAML>=6,<7"')
import importlib.metadata as md
for package in ("credsweeper", "torch", "transformers", "peft", "numpy"):
    print(package, md.version(package))
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## ④ gitleaks

서버와 같은 8.30.1. 없으면 플러그인이 gitleaks 모듈만 빼고 돌아 서버 결과와 비교할 수 없다.

In [ ]:
GITLEAKS = "8.30.1"
if subprocess.run("gitleaks version", shell=True, capture_output=True).returncode != 0:
    sh(f"curl -sSL https://github.com/gitleaks/gitleaks/releases/download/v{GITLEAKS}/gitleaks_{GITLEAKS}_linux_x64.tar.gz"
       " | tar -xz -C /usr/local/bin gitleaks")
sh("gitleaks version")

## ⑤ 데이터

압축 파일에는 `data_test/`와 `data_answer/`가 들어 있다. `promptguard-data/metric-detection/` 아래에 푼다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

MD = "/content/promptguard-data/metric-detection"
sh(f"rm -rf {MD}/data_test {MD}/data_answer && tar -xzf {shlex.quote(DATA_ARCHIVE)} -C {MD}")
sh(f"ls -la {MD}/data_test {MD}/data_answer")

## ⑥ 작은 확인

예시 데이터로 ML 모듈이 **실제로 켜지는지** 본다. `--require-adapters`가 있어서 패키지나 가중치 문제로 모듈이 빠지면 이유를 출력하고 멈춘다(종료 코드 3). Qwen은 처음 실행할 때 베이스 모델(약 1GB)을 받는다.

In [ ]:
import glob, json, sys
from pathlib import Path

REPO = "/content/promptguard-data"
SYSTEM_ROOT = "/content/promptguard-claude-demoV0"
RESULTS = "/content/results"

if V9_DEVICE:  # 이 Colab 사본의 detector.json만 바꾼다
    config_path = Path(SYSTEM_ROOT, "src/detector/detector.json")
    config = json.loads(config_path.read_text())
    for adapter in config["adapters"]:
        if adapter["name"] == "ml-v9":
            adapter["options"]["device"] = V9_DEVICE
    config_path.write_text(json.dumps(config, indent=2))
    print("ml-v9 device:", V9_DEVICE)

def evaluate(sessions, out, enable):
    env = {key: value for key, value in os.environ.items() if not key.startswith("PG_")}
    command = [sys.executable, "metric-detection/scripts/evaluate.py", "run", "--system", "promptguard",
               "--system-root", SYSTEM_ROOT, "--use", "ml_eval", "--on-detector-failure", ON_FAILURE,
               "--sessions", sessions, "--out", out]
    if enable:
        env["PG_DETECTOR_ENABLE"] = enable
        command += ["--require-adapters", enable]
    print("$", ("PG_DETECTOR_ENABLE=" + enable + " ") if enable else "", shlex.join(command), flush=True)
    completed = subprocess.run(command, cwd=REPO, env=env)
    if completed.returncode != 0:
        raise RuntimeError(f"evaluate.py exit {completed.returncode}")
    return max(glob.glob(f"{out}/*_promptguard"), key=os.path.getmtime)

run = evaluate(f"{MD}/data_test/sessions_from_example.jsonl", f"{RESULTS}/smoke", ENABLE)
meta = json.load(open(f"{run}/run_meta.json"))["system"]
print("adapters:", [adapter["name"] for adapter in meta["detector"]["adapters"]])
print("skipped:", meta["detector"]["skipped"])
print("blocked_items:", meta["blocked_items"], "failed_open_items:", meta["failed_open_items"])

## ⑦ 전체 평가

데이터셋마다 한 번씩 돌린다. `RUN_BASELINE`이면 ML 없이 한 번 더 돌린다. 시간 감: ml-v9는 입력당 수십 ms라 1만 개 표본은 수십 분 정도 걸린다. ml-qwen은 단서 단어가 있는 입력만 보지만 훨씬 느리다. 셀이 끊겨도 끝난 데이터셋은 `RESULTS` 아래에 남는다. 다시 실행하면 이미 끝난 조합은 건너뛴다.

In [ ]:
datasets = sorted(Path(MD, "data_test").glob("sessions*.jsonl"))
if DATASETS:
    datasets = [path for path in datasets if path.stem in DATASETS]
configs = [("ml", ENABLE)] + ([("baseline", "")] if RUN_BASELINE else [])

runs = {}
for label, enable in configs:
    for path in datasets:
        out = f"{RESULTS}/{label}/{path.stem}"
        done = sorted(glob.glob(f"{out}/*_promptguard/metrics.json"))
        if done:
            runs[label, path.stem] = str(Path(done[-1]).parent)
            print(f"skip {label} {path.stem}: done")
            continue
        runs[label, path.stem] = evaluate(str(path), out, enable)

## ⑧ 결과 표와 저장

- **recall / precision / F2**: 비밀값만 정확히 가렸는지 기준이다.
- **blocked**: 탐지기 오류(ML timeout 등)로 Final Guard가 막았다고 본 item 수다. 0이 아니면 ML 모듈이 그 입력을 처리하지 못한 것이다. 그 item의 정답은 `over`로 채점된다.
- **ML 모듈이 빠지면** 그 실행은 이미 ⑥·⑦에서 멈췄다. 표의 adapters 열로 다시 확인할 수 있다.

In [ ]:
import pandas as pd

rows = []
for (label, stem), run in sorted(runs.items()):
    metrics = json.load(open(f"{run}/metrics.json"))
    system = json.load(open(f"{run}/run_meta.json"))["system"]
    rows.append({"config": label, "dataset": stem.removeprefix("sessions_").removeprefix("from_"),
                 "items": metrics["items"], "gold": metrics["counts"]["gold_total"],
                 "recall": metrics["recall"], "precision": metrics["precision"], "f2": metrics["f2"],
                 "fp_per_1k_lines": metrics["fp_per_1k_lines"],
                 "latency_p50_ms": (metrics.get("latency_ms") or {}).get("p50"),
                 "blocked": system["blocked_items"], "failed_open": system["failed_open_items"],
                 "adapters": ",".join(a["name"] for a in system["detector"]["adapters"]), "run": run})
table = pd.DataFrame(rows)
display(table.drop(columns=["run"]).round(4))
if RUN_BASELINE:
    wide = table.pivot(index="dataset", columns="config", values=["recall", "precision", "f2"])
    display(wide.round(4))

os.makedirs(OUT_DRIVE, exist_ok=True)
stamp = __import__("datetime").datetime.now().strftime("%Y%m%d-%H%M%S")
table.to_csv(f"{RESULTS}/summary.csv", index=False)
sh(f"cp -r {RESULTS} {shlex.quote(OUT_DRIVE)}/colab_{stamp}")
print("saved:", f"{OUT_DRIVE}/colab_{stamp}")